# Final Python Steps — Renewable Energy Analytics
No SQL or Tableau is needed. Run these cells after your main analysis has created `generation`, `weather`, `merged`, `daily_production`, `plant_analysis`, and `low_output`.

## 1. Validate outputs

In [ ]:
required = ["generation", "weather", "merged", "daily_production", "plant_analysis", "low_output"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Run the earlier analysis cells first. Missing: {missing}")

print("Generation:", generation.shape)
print("Weather:", weather.shape)
print("Merged:", merged.shape)

## 2. Validate merged data

In [ ]:
print("Missing values:")
display(merged.isna().sum().to_frame("missing_values"))
print("Records by plant:")
display(merged.groupby("PLANT_ID").size().to_frame("records"))
print("Duplicate plant/timestamp keys:", merged.duplicated(["PLANT_ID", "DATE_TIME"]).sum())
print("Date range:", merged["DATE_TIME"].min(), "to", merged["DATE_TIME"].max())

## 3. Create an hourly summary

In [ ]:
hourly_summary = (
    generation.assign(HOUR=generation["DATE_TIME"].dt.hour)
    .groupby(["PLANT_ID", "HOUR"], as_index=False)
    .agg(
        average_ac_power_kw=("AC_POWER", "mean"),
        maximum_ac_power_kw=("AC_POWER", "max"),
        readings=("AC_POWER", "count")
    )
)
display(hourly_summary.head(10))

## 4. Create weather correlation table

In [ ]:
candidate_columns = [
    "AC_POWER", "DC_POWER", "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE", "IRRADIATION"
]
available_columns = [c for c in candidate_columns if c in merged.columns]
weather_correlations = merged[available_columns].corr(numeric_only=True)
display(weather_correlations.round(3))

## 5. Save charts as PNG files

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

chart_dir = Path("charts")
chart_dir.mkdir(exist_ok=True)

ax = plant_analysis.plot(
    x="PLANT_ID",
    y=["Average_AC_Power", "Maximum_AC_Power"],
    kind="bar",
    figsize=(9, 5)
)
ax.set_title("Solar Power Generation by Plant")
ax.set_xlabel("Plant ID")
ax.set_ylabel("Power (kW)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(chart_dir / "plant_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

hourly = (
    generation.assign(HOUR=generation["DATE_TIME"].dt.hour)
    .groupby(["PLANT_ID", "HOUR"])["AC_POWER"].mean().reset_index()
)
plt.figure(figsize=(11, 5))
for plant_id, data in hourly.groupby("PLANT_ID"):
    plt.plot(data["HOUR"], data["AC_POWER"], marker="o", label=f"Plant {plant_id}")
plt.title("Average AC Power by Hour")
plt.xlabel("Hour")
plt.ylabel("Average AC Power (kW)")
plt.xticks(range(24))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(chart_dir / "hourly_power.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(12, 5))
for plant_id, data in daily_production.groupby("PLANT_ID"):
    plt.plot(data["DATE"], data["Estimated_Daily_Energy"], marker="o", label=f"Plant {plant_id}")
plt.title("Daily Solar Yield")
plt.xlabel("Date")
plt.ylabel("Daily yield (verify source units)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(chart_dir / "daily_yield.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

print("Saved chart files:", [p.name for p in chart_dir.glob("*.png")])

## 6. Generate a findings report from your actual metrics

In [ ]:
from pathlib import Path

report_dir = Path("reports")
report_dir.mkdir(exist_ok=True)

report_lines = [
    "# Renewable Energy Production & Efficiency Analytics",
    "",
    "## Dataset overview",
    f"- Generation records: {len(generation):,}",
    f"- Weather records: {len(weather):,}",
    f"- Plants analyzed: {generation['PLANT_ID'].nunique()}",
    f"- Date range: {generation['DATE_TIME'].min()} to {generation['DATE_TIME'].max()}",
    "",
    "## Plant summary",
    plant_analysis.round(2).to_string(index=False),
    "",
    "## Findings to complete",
    "1. Describe the hourly production pattern using the chart.",
    "2. Compare plants/inverters while considering possible capacity differences.",
    "3. Report weather correlations and explain that correlation is not causation.",
    f"4. Potential low-output observations flagged: {len(low_output)}.",
    "5. Add one evidence-based recommendation.",
    "",
    "## Limitations",
    "- Dataset covers a short period; avoid annual seasonality claims.",
    "- AC_POWER/DC_POWER are power readings, not energy totals.",
    "- DAILY_YIELD is a counter; verify source units and missing end-of-day readings.",
    "- Low-output flags are not confirmed equipment failures."
]
report_path = report_dir / "key_findings.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(report_path.read_text())

## 7. Export all tables, charts, and report

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

output_dir = Path("renewable_energy_project_outputs")
output_dir.mkdir(exist_ok=True)

tables = {
    "cleaned_generation_data.csv": generation,
    "cleaned_weather_data.csv": weather,
    "generation_weather_merged.csv": merged,
    "daily_production.csv": daily_production,
    "plant_summary.csv": plant_analysis,
    "hourly_summary.csv": hourly_summary,
    "weather_power_correlations.csv": weather_correlations,
    "potential_low_output.csv": low_output
}
for filename, df in tables.items():
    df.to_csv(output_dir / filename, index=False)

shutil.copytree("charts", output_dir / "charts", dirs_exist_ok=True)
shutil.copytree("reports", output_dir / "reports", dirs_exist_ok=True)
shutil.make_archive("renewable_energy_project_outputs", "zip", output_dir)
files.download("renewable_energy_project_outputs.zip")

## GitHub checklist
- Save your completed Colab notebook as `Renewable_Energy_Analytics.ipynb`.
- Include the README, findings report, and selected chart PNGs.
- Do not upload raw dataset files unless the license permits redistribution.
- Make sure your final findings reflect your actual analysis results.